# Phase 2 — Step 1 v3: Population AF Filter + Control Presence Features

**Project:** ML-Based Pathogenicity Prediction of CRISPR Off-Target Variants in HSPCs for Beta-Thalassemia Therapy

**What this notebook does:**
1. Uploads the full GEMINI query output (~4 million variants, all 8 samples)
2. Keeps ALL variants present in ≥1 edited sample (no hard germline subtraction)
3. Applies population AF filter — keeps novel (AF=-1) or rare (global AF < 0.01)
4. Adds `in_controls` and `num_control_samples` as ML features instead of hard-filtering
5. Saves the filtered candidate variant list as CSV for downstream steps

**Why no hard germline subtraction:**
- Edited and control HSPCs come from the **same donors** — shared variants may be
  pre-existing disease-associated mutations in beta-thalassemia patients, not just noise
- Hard subtraction removed all 22 original Phase 1 candidates including HIGH-impact
  splice variants (SPICE1 CADD=25.8, TNRC18 CADD=22.2) — too aggressive
- Instead, `in_controls` and `num_control_samples` are added as ML features so
  the model learns whether control presence affects pathogenicity prediction
- This is scientifically more appropriate for CRISPR base-editing in disease samples

**Sample order (confirmed from GEMINI `samples` table):**

| gt_types position | Sample name | Type |
|---|---|---|
| 0 | donor1_ctrl1 | CONTROL |
| 1 | donor1_edit | EDITED |
| 2 | donor2_ctrl | CONTROL |
| 3 | donor2_edit | EDITED |
| 4 | donor3_ctrl | CONTROL |
| 5 | donor3_edit | EDITED |
| 6 | hspc_edit | EDITED |
| 7 | hspc_ctrl | CONTROL |

**Edited positions: 1, 3, 5, 6 | Control positions: 0, 2, 4, 7**

## Cell 1 — Install Libraries

In [ ]:
!pip install pandas numpy tqdm --quiet
print('✅ Libraries ready')

## Cell 2 — Mount Google Drive & Locate File

Mount your Google Drive. Your bz2 file should be in the root of `My Drive`:
`Galaxy151-[LL_VARIANTS_ALL_SAMPLES GEMINI query on dataset 133].tabular.bz2`

✅ pandas reads bz2 natively — no manual decompression needed.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path to your bz2 file in root of My Drive
filename = '/content/drive/My Drive/Galaxy151-[LL_VARIANTS_ALL_SAMPLES GEMINI query on dataset 133].tabular.bz2'

# Verify file exists
if os.path.exists(filename):
    size_mb = os.path.getsize(filename) / 1e6
    print(f'✅ File found: {filename}')
    print(f'   Size: {size_mb:.1f} MB')
    print(f'   pandas will decompress bz2 automatically during loading')
else:
    print('❌ File not found! Check the path.')
    print('   Listing My Drive root:')
    for f in os.listdir('/content/drive/My Drive/'):
        print(f'   {f}')

## Cell 3 — Load Data in Chunks

Reads the bz2 file directly from Google Drive in 500K-row chunks.
Keeps ALL variants present in ≥1 edited sample + AF filter only.
Expected output: ~1.5–2M variants. Runtime: 4–8 minutes.

In [ ]:
import pandas as pd
import numpy as np

# ── Sample position maps (confirmed from GEMINI samples table) ──
EDITED_POS  = [1, 3, 5, 6]   # donor1_edit, donor2_edit, donor3_edit, hspc_edit
CONTROL_POS = [0, 2, 4, 7]   # donor1_ctrl1, donor2_ctrl, donor3_ctrl, hspc_ctrl
ABSENT = {0, 2}              # HOM_REF or MISSING = absent

CHUNKSIZE = 500_000

def parse_gt(s):
    return [int(x) for x in str(s).split(',')]

def in_edited(gt):
    """Keep if present (HET or HOM_ALT) in at least 1 edited sample."""
    return any(gt[i] not in ABSENT for i in EDITED_POS)

def count_control_samples(gt):
    """Count how many control samples carry this variant."""
    return sum(1 for i in CONTROL_POS if gt[i] not in ABSENT)

def count_edited_samples(gt):
    """Count how many edited samples carry this variant."""
    return sum(1 for i in EDITED_POS if gt[i] not in ABSENT)

def max_edited_gt_val(gt):
    """Highest gt value among edited samples (1=HET, 3=HOM_ALT)."""
    vals = [gt[i] for i in EDITED_POS if gt[i] not in ABSENT]
    return max(vals) if vals else 0

print('Loading bz2 file from Google Drive in chunks...')
print(f'File      : {filename}')
print(f'Chunk size: {CHUNKSIZE:,} rows')
print(f'Filter    : present in >=1 edited sample + global AF < 1% or novel')
print(f'NO hard germline subtraction — control presence added as ML feature')
print('-' * 65)

chunks_filtered = []
total_rows = 0
total_kept = 0

reader = pd.read_csv(
    filename,
    sep='\t',
    compression='bz2',
    chunksize=CHUNKSIZE,
    low_memory=False
)

for i, chunk in enumerate(reader):
    total_rows += len(chunk)

    # Parse gt_types
    chunk['gt_list'] = chunk['gt_types'].apply(parse_gt)

    # Filter 1: present in at least 1 edited sample
    mask_edited = chunk['gt_list'].apply(in_edited)

    # Filter 2: population AF — novel (-1) or rare (< 1%)
    mask_af = (chunk['max_aaf_all'] == -1) | \
              ((chunk['max_aaf_all'] >= 0) & (chunk['max_aaf_all'] < 0.01))

    filtered = chunk[mask_edited & mask_af].copy()

    # Add control presence features
    filtered['num_control_samples'] = filtered['gt_list'].apply(count_control_samples)
    filtered['in_controls']         = (filtered['num_control_samples'] > 0).astype(int)
    filtered['num_edited_samples']  = filtered['gt_list'].apply(count_edited_samples)
    filtered['max_edited_gt']       = filtered['gt_list'].apply(max_edited_gt_val)

    filtered.drop(columns=['gt_list'], inplace=True)
    chunks_filtered.append(filtered)
    total_kept += len(filtered)

    print(f'  Chunk {i+1}: {len(chunk):>7,} rows → {len(filtered):>6,} kept  '
          f'(running total: {total_rows:,} read, {total_kept:,} kept)')

print()
print('Combining chunks...')
df = pd.concat(chunks_filtered, ignore_index=True)

print(f'\n✅ Done!')
print(f'   Total rows read : {total_rows:>10,}')
print(f'   Final kept      : {len(df):>10,}')

## Cell 4 — Preview & Basic Stats

In [ ]:
import pandas as pd

print('FILTERED VARIANT DATASET — OVERVIEW')
print('='*60)
print(f'Total variants : {len(df):,}')
print(f'Columns        : {list(df.columns)}')
print()

print('── Impact Severity Breakdown ──')
impact_counts = df['impact_severity'].value_counts()
for lvl, cnt in impact_counts.items():
    pct = cnt / len(df) * 100
    print(f'  {lvl:<10}: {cnt:>8,}  ({pct:.2f}%)')
print()

novel = (df['max_aaf_all'] == -1).sum()
rare  = ((df['max_aaf_all'] >= 0) & (df['max_aaf_all'] < 0.01)).sum()
print('── Population AF Breakdown ──')
print(f'  Novel  (AF=-1) : {novel:>8,}  ({novel/len(df)*100:.2f}%)')
print(f'  Rare   (AF<1%) : {rare:>8,}  ({rare/len(df)*100:.2f}%)')
print()

print('── Control Presence (ML Feature) ──')
in_ctrl = df['in_controls'].sum()
not_ctrl = (df['in_controls']==0).sum()
print(f'  Also in ≥1 control : {in_ctrl:>8,}  ({in_ctrl/len(df)*100:.2f}%) — kept, used as feature')
print(f'  Edited-only        : {not_ctrl:>8,}  ({not_ctrl/len(df)*100:.2f}%)')
print()

clinvar_found = df['clinvar_sig'].notna() & (df['clinvar_sig'] != 'None') & (df['clinvar_sig'] != '')
print('── ClinVar Hits ──')
print(f'  Variants with ClinVar annotation: {clinvar_found.sum():,}')
print()

print('── Score Availability ──')
for col in ['cadd_phred', 'gerp_rs', 'cadd_scaled', 'gerp_bp_score']:
    if col in df.columns:
        print(f'  {col}: {df[col].notna().sum():,}  ({df[col].notna().mean()*100:.1f}%)')
print()

print('── First 5 Rows ──')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
display(df.head())

## Cell 5 — Add Derived Columns

Add useful columns for downstream ML steps:
- `novelty_class`: Novel / Rare
- `is_novel`: binary flag (1 = novel, 0 = rare)
- `is_splice`: binary flag for splice variants
- `variant_type`: clean impact string for ML feature engineering
- `num_edited_samples`: how many edited samples carry this variant
- `max_edited_gt`: highest gt_type among edited samples (HET=1 vs HOM_ALT=3)

In [ ]:
import numpy as np

# Novelty class
df['novelty_class'] = np.where(df['max_aaf_all'] == -1, 'Novel', 'Rare')
df['is_novel']      = (df['max_aaf_all'] == -1).astype(int)

# Variant type
df['variant_type'] = df['impact'].str.lower()

# Splice flag
SPLICE_KEYWORDS = ['splice_donor', 'splice_acceptor', 'splice_region']
df['is_splice'] = df['variant_type'].apply(
    lambda x: int(any(k in str(x) for k in SPLICE_KEYWORDS))
)

# Germline flag (for reference — variants shared with controls)
df['is_germline_candidate'] = df['in_controls'].copy()

# Rename score columns to match downstream notebooks
df.rename(columns={
    'start'        : 'pos',
    'cadd_scaled'  : 'cadd_phred',
    'gerp_bp_score': 'gerp_rs'
}, inplace=True, errors='ignore')

print('✅ Derived columns added')
print(f'   Columns now: {list(df.columns)}')
print()
print('Novelty breakdown:')
print(df['novelty_class'].value_counts())
print()
print('Splice variants:', df['is_splice'].sum())
print()
print('Control presence breakdown (in_controls):')
print(df['in_controls'].value_counts())
print(f'  → {df["in_controls"].sum():,} variants also present in ≥1 control (kept as ML feature)')
print(f'  → {(df["in_controls"]==0).sum():,} variants unique to edited samples')
print()
print('Edited sample count distribution:')
print(df['num_edited_samples'].value_counts().sort_index())

## Cell 6 — Visualise Filtered Variant Distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Filtered CRISPR Off-Target Variant Dataset\n'
             '(AF < 1% / Novel, Control Presence as Feature)', fontsize=13, fontweight='bold')

# Plot 1: Impact severity
ax = axes[0]
impact_order  = ['HIGH', 'MED', 'LOW']
impact_colors = {'HIGH': '#C0392B', 'MED': '#E67E22', 'LOW': '#27AE60'}
impact_vals   = [df['impact_severity'].value_counts().get(k, 0) for k in impact_order]
bars = ax.bar(impact_order, impact_vals,
              color=[impact_colors[k] for k in impact_order], edgecolor='white', width=0.5)
for bar, val in zip(bars, impact_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(impact_vals)*0.01,
            f'{val:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title('Impact Severity', fontweight='bold')
ax.set_ylabel('Variant Count')
ax.set_ylim(0, max(impact_vals)*1.15)
ax.spines[['top','right']].set_visible(False)

# Plot 2: Novelty
ax = axes[1]
nov_counts = df['novelty_class'].value_counts()
ax.pie(nov_counts.values, labels=nov_counts.index,
       autopct='%1.1f%%', colors=['#2E5FA3','#E67E22'][:len(nov_counts)],
       startangle=90, textprops={'fontsize':10})
ax.set_title('Population Novelty', fontweight='bold')

# Plot 3: Control presence
ax = axes[2]
ctrl_labels = ['Edited-only\n(in_controls=0)', 'Also in controls\n(in_controls=1)']
ctrl_vals   = [(df['in_controls']==0).sum(), df['in_controls'].sum()]
ctrl_colors = ['#2E5FA3', '#E74C3C']
bars = ax.bar(ctrl_labels, ctrl_vals, color=ctrl_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, ctrl_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(ctrl_vals)*0.01,
            f'{val:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title('Control Presence\n(ML Feature)', fontweight='bold')
ax.set_ylabel('Variant Count')
ax.set_ylim(0, max(ctrl_vals)*1.15)
ax.spines[['top','right']].set_visible(False)

# Plot 4: Num edited samples
ax = axes[3]
ns_counts = df['num_edited_samples'].value_counts().sort_index()
bars = ax.bar(ns_counts.index.astype(str), ns_counts.values,
              color='#2E5FA3', edgecolor='white', width=0.6)
for bar, val in zip(bars, ns_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+ns_counts.max()*0.01,
            f'{val:,}', ha='center', va='bottom', fontsize=7, fontweight='bold')
ax.set_title('Variants by # Edited Samples', fontweight='bold')
ax.set_xlabel('Number of edited samples')
ax.set_ylabel('Variant Count')
ax.set_ylim(0, ns_counts.max()*1.15)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
os.makedirs('figures_step1', exist_ok=True)
plt.savefig('figures_step1/variant_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure saved')

## Cell 7 — Chromosome Distribution

In [ ]:
import matplotlib.pyplot as plt
import re

# Sort chromosomes numerically
def chrom_sort_key(c):
    c = str(c).replace('chr', '')
    if c == 'X': return 23
    if c == 'Y': return 24
    if c == 'M' or c == 'MT': return 25
    try: return int(c)
    except: return 99

chrom_counts = df['chrom'].value_counts()
chrom_counts = chrom_counts.reindex(
    sorted(chrom_counts.index, key=chrom_sort_key)
)

fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(range(len(chrom_counts)), chrom_counts.values,
              color='#2E5FA3', edgecolor='white', width=0.7)
ax.set_xticks(range(len(chrom_counts)))
ax.set_xticklabels(chrom_counts.index, rotation=45, ha='right', fontsize=9)
ax.set_title('Filtered Variants by Chromosome\n(Edited-Only, AF < 1% or Novel)',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Variant Count')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('figures_step1/chromosome_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure saved: figures_step1/chromosome_distribution.png')

## Cell 8 — HIGH Impact Variants Spotlight

Shows all HIGH impact variants in the filtered set — these are the most critical candidates.

In [ ]:
import pandas as pd

high = df[df['impact_severity'] == 'HIGH'].copy()
high = high.sort_values('cadd_phred', ascending=False)

print(f'HIGH IMPACT VARIANTS — {len(high):,} total')
print('='*80)

display_cols = ['chrom','pos','ref','alt','gene','impact','cadd_phred',
                'gerp_rs','max_aaf_all','novelty_class',
                'num_edited_samples','clinvar_sig']

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
display(high[display_cols].head(50).reset_index(drop=True))

print()
print('Impact type breakdown within HIGH:')
print(high['impact'].value_counts().head(20))
print()
print('HIGH variants with ClinVar annotation:')
high_clinvar = high[high['clinvar_sig'].notna() &
                    (high['clinvar_sig'] != 'None') &
                    (high['clinvar_sig'] != '')]
print(f'  {len(high_clinvar)} variants')
if len(high_clinvar) > 0:
    display(high_clinvar[display_cols].head(20).reset_index(drop=True))

## Cell 9 — Verify the Original 22 Candidates Are Present

Cross-checks that the 22 variants from Phase 1 (your PI's original results) are captured in this larger filtered set. This validates the filtering logic is not too aggressive.

In [ ]:
import pandas as pd

# Original 22 variants — positions are 1-based from the PDF
# GEMINI stores positions 0-based, so we check pos-1
original_22 = [
    ('chr1',  21072082, 'A',  'G',   'HP1BP3'),
    ('chr10', 46342157, 'G',  'T',   'AGAP4'),
    ('chr12', 394677,   'G',  'C',   'KDM5A'),
    ('chr12', 40044027, 'AT', 'A',   'C12orf40'),
    ('chr14', 97022168, 'AT', 'ATT', 'PAPOLA'),
    ('chr15', 41709326, 'C',  'A',   'RTF1'),
    ('chr16', 8988686,  'C',  'G',   'USP7'),
    ('chr18', 43469840, 'C',  'G',   'EPG5'),
    ('chr2',  46839487, 'GA', 'G',   'PIGF'),
    ('chr3',  113218285,'C',  'G',   'SPICE1'),
    ('chr3',  113218285,'C',  'A',   'SPICE1'),
    ('chr3',  170727846,'T',  'C',   'SLC2A2'),
    ('chr4',  367275,   'A',  'G',   'ZNF141'),
    ('chr4',  189020278,'C',  'T',   'TRIML2'),
    ('chr4',  189020278,'C',  'G',   'TRIML2'),
    ('chr5',  65473406, 'G',  'A',   'SREK1'),
    ('chr5',  65473406, 'G',  'C',   'SREK1'),
    ('chr5',  131927550,'AT', 'ATT', 'RAD50'),
    ('chr6',  88224021, 'C',  'A',   'RARS2'),
    ('chr7',  5385191,  'A',  'C',   'TNRC18'),
    ('chr7',  87516629, 'CT', 'C',   'DBF4'),
    ('chr8',  86022336, 'CT', 'C',   'LRRCC1'),
]

print('CROSS-CHECKING ORIGINAL 22 VARIANTS (trying 1-based and 0-based positions)')
print('='*70)

found = 0
for chrom, pos1, ref, alt, gene in original_22:
    pos0 = pos1 - 1
    # Try 1-based first, then 0-based
    hit = df[
        (df['chrom']==chrom) &
        (df['pos'].isin([pos0, pos1])) &
        (df['ref']==ref) & (df['alt']==alt)
    ]
    if len(hit) > 0:
        found += 1
        row = hit.iloc[0]
        ctrl_info = f'in_controls={row["in_controls"]} ({row["num_control_samples"]} ctrl samples)'
        print(f'  ✅ {gene:10s}  {chrom}:{row["pos"]}  {ref}>{alt}  '
              f'impact={row["impact_severity"]}  {ctrl_info}')
    else:
        print(f'  ❌ {gene:10s}  {chrom}:{pos1}  {ref}>{alt}  — NOT FOUND (may be filtered by AF)')

print()
print(f'Found: {found}/22 original variants in filtered set')
if found == 22:
    print('✅ All 22 original variants recovered!')
elif found > 0:
    print(f'⚠️  {22-found} variants missing — likely AF >= 0.01 in max_aaf_all')

## Cell 10 — Save Filtered Variants as CSV

Saves the full filtered variant set as `all_variants_filtered_step1.csv`.
This is the input file for **Step 2 (ClinVar Training Data)** and **Step 3 (Enrichment)**.

In [ ]:
from google.colab import files
import os

save_cols = [
    'chrom', 'pos', 'ref', 'alt', 'gene',
    'impact', 'impact_severity', 'variant_type',
    'cadd_phred', 'gerp_rs',
    'max_aaf_all', 'novelty_class', 'is_novel',
    'is_splice', 'rs_ids',
    'clinvar_sig', 'clinvar_disease_name',
    'num_edited_samples', 'max_edited_gt',
    'in_controls', 'num_control_samples',
    'is_germline_candidate',
    'gt_types', 'gts', 'gt_depths', 'gt_alt_freqs'
]
save_cols = [c for c in save_cols if c in df.columns]

out_path = 'all_variants_filtered_step1.csv'
df[save_cols].to_csv(out_path, index=False)

size_mb = os.path.getsize(out_path) / 1e6
print(f'✅ Saved: {out_path}')
print(f'   Rows   : {len(df):,}')
print(f'   Columns: {len(save_cols)} — includes in_controls & num_control_samples as ML features')
print(f'   Size   : {size_mb:.1f} MB')
print()
files.download(out_path)
print(f'📥 {out_path} — use this file in Step 3 (Enrichment)')

## Cell 11 — Download Figures

In [ ]:
from google.colab import files
import os

for f in os.listdir('figures_step1'):
    fpath = f'figures_step1/{f}'
    files.download(fpath)
    print(f'📥 {f}')

print()
print('STEP 1 COMPLETION SUMMARY')
print('='*60)
print(f'Input    : Full GEMINI output (~4M variants, 8 samples)')
print(f'Filtered : {len(df):,} variants')
print()
print('Filters applied:')
print('  1. Edited sample filter — kept variants present in')
print('     >=1 edited sample (HET or HOM_ALT)')
print('     NO hard germline subtraction — same-donor cells')
print()
print('  2. Population AF filter:')
print('     max_aaf_all == -1 (novel) OR max_aaf_all < 0.01 (rare)')
print()
print('New ML features added:')
print('  in_controls         : 1 if variant also in >=1 control')
print('  num_control_samples : how many controls carry it (0-4)')
print('  is_germline_candidate: same as in_controls')
print()
print('Output   : all_variants_filtered_step1.csv')
print('Next step: Step 2 — ClinVar Training Data (Oct 2024 snapshot)')